# Otimização de Custos em Mistura de Concreto

Grupo 06 — Luiz Gabriel e Gabriel Lima

Este notebook reproduz a lógica da planilha com Solver: minimizar o custo de uma mistura de concreto para 1 m³, respeitando restrições de volume, resistência, relação água/cimento e proporções viáveis entre materiais.

> Observação: modelo acadêmico simplificado, baseado em premissas técnicas e econômicas. Não substitui dosagem experimental em laboratório.

## Ajuste importante
A planilha considera uma parcela de **ar/vazios de 0,02 m³** no cálculo do volume absoluto. Por isso, o Python também considera esse termo para ficar compatível com a planilha.


In [1]:
import pandas as pd

# -----------------------------
# Dados do modelo
# -----------------------------

precos = {
    "cimento": 0.6600,   # R$/kg
    "agua": 0.0164,      # R$/L
    "areia": 0.0531,     # R$/kg
    "brita": 0.0519      # R$/kg
}

massa_especifica = {
    "cimento": 3150,  # kg/m³
    "agua": 1000,     # kg/m³
    "areia": 2650,    # kg/m³
    "brita": 2700     # kg/m³
}

# Parcela de ar/vazios considerada na planilha
volume_ar = 0.02  # m³

# Parâmetros de resistência
fck_referencia = 25
ac_referencia = 0.55
expoente_empirico = 1.70

# Restrições do modelo
restricoes = {
    "volume_min": 0.995,
    "volume_max": 1.005,
    "ac_max": 0.55,
    "fck_min": 25,
    "areia_cimento_min": 2.07,
    "areia_cimento_max": 2.53,
    "brita_cimento_min": 2.43,
    "brita_cimento_max": 2.97,
    "cimento_min": 280,
    "cimento_max": 430,
    "agua_min": 160,
    "agua_max": 200,
    "areia_min": 700,
    "areia_max": 900,
    "brita_min": 900,
    "brita_max": 1100
}

# Tolerância para evitar falso "FORA" por arredondamento numérico
EPS = 1e-4


In [2]:
def calcular_indicadores(cimento, agua, areia, brita):
    """Calcula custo, volume, relação água/cimento, fck e proporções."""

    custo = (
        cimento * precos["cimento"] +
        agua * precos["agua"] +
        areia * precos["areia"] +
        brita * precos["brita"]
    )

    volume = (
        cimento / massa_especifica["cimento"] +
        agua / massa_especifica["agua"] +
        areia / massa_especifica["areia"] +
        brita / massa_especifica["brita"] +
        volume_ar
    )

    ac = agua / cimento
    fck = fck_referencia * (ac_referencia / ac) ** expoente_empirico

    areia_cimento = areia / cimento
    brita_cimento = brita / cimento

    return {
        "Custo total (R$/m³)": custo,
        "Volume calculado (m³)": volume,
        "Relação água/cimento": ac,
        "fck estimado (MPa)": fck,
        "Areia/cimento": areia_cimento,
        "Brita/cimento": brita_cimento
    }


def verificar_restricoes(cimento, agua, areia, brita):
    """Verifica se uma mistura atende às restrições."""

    ind = calcular_indicadores(cimento, agua, areia, brita)

    checks = {
        "Volume mínimo": ind["Volume calculado (m³)"] >= restricoes["volume_min"] - EPS,
        "Volume máximo": ind["Volume calculado (m³)"] <= restricoes["volume_max"] + EPS,
        "Água/cimento máximo": ind["Relação água/cimento"] <= restricoes["ac_max"] + EPS,
        "fck mínimo": ind["fck estimado (MPa)"] >= restricoes["fck_min"] - EPS,
        "Areia/cimento mínimo": ind["Areia/cimento"] >= restricoes["areia_cimento_min"] - EPS,
        "Areia/cimento máximo": ind["Areia/cimento"] <= restricoes["areia_cimento_max"] + EPS,
        "Brita/cimento mínimo": ind["Brita/cimento"] >= restricoes["brita_cimento_min"] - EPS,
        "Brita/cimento máximo": ind["Brita/cimento"] <= restricoes["brita_cimento_max"] + EPS,
        "Cimento mínimo": cimento >= restricoes["cimento_min"] - EPS,
        "Cimento máximo": cimento <= restricoes["cimento_max"] + EPS,
        "Água mínima": agua >= restricoes["agua_min"] - EPS,
        "Água máxima": agua <= restricoes["agua_max"] + EPS,
        "Areia mínima": areia >= restricoes["areia_min"] - EPS,
        "Areia máxima": areia <= restricoes["areia_max"] + EPS,
        "Brita mínima": brita >= restricoes["brita_min"] - EPS,
        "Brita máxima": brita <= restricoes["brita_max"] + EPS,
    }

    return ind, checks


## 1. Avaliação da mistura final usada na planilha

Valores finais da planilha:

- Cimento: 350 kg
- Água: 190 L
- Areia: 780 kg
- Brita: 1030 kg


In [3]:
mistura_planilha = {
    "cimento": 350,
    "agua": 190,
    "areia": 780,
    "brita": 1030
}

ind_planilha, checks_planilha = verificar_restricoes(**mistura_planilha)

df_planilha = pd.DataFrame([
    ["Cimento CP II-32", mistura_planilha["cimento"], "kg"],
    ["Água", mistura_planilha["agua"], "L"],
    ["Areia média", mistura_planilha["areia"], "kg"],
    ["Brita 1", mistura_planilha["brita"], "kg"],
], columns=["Material", "Quantidade", "Unidade"])

df_ind_planilha = pd.DataFrame(ind_planilha.items(), columns=["Indicador", "Valor"])
df_check_planilha = pd.DataFrame(
    [(k, "OK" if v else "FORA") for k, v in checks_planilha.items()],
    columns=["Restrição", "Atendida?"]
)

print("=== Mistura da planilha ===")
display(df_planilha)
display(df_ind_planilha.round(6))
display(df_check_planilha)

if all(checks_planilha.values()):
    print("Resultado da planilha: VIÁVEL.")
else:
    print("Resultado da planilha: NÃO VIÁVEL.")


=== Mistura da planilha ===


,Material,Quantidade,Unidade
0,Cimento CP II-32,350,kg
1,Água,190,L
2,Areia média,780,kg
3,Brita 1,1030,kg


,Indicador,Valor
0,Custo total (R$/m³),328.991000
1,Volume calculado (m³),0.996932
2,Relação água/cimento,0.542857
3,fck estimado (MPa),25.561782
4,Areia/cimento,2.228571
5,Brita/cimento,2.942857


,Restrição,Atendida?
0,Volume mínimo,OK
1,Volume máximo,OK
2,Água/cimento máximo,OK
3,fck mínimo,OK
4,Areia/cimento mínimo,OK
5,Areia/cimento máximo,OK
6,Brita/cimento mínimo,OK
7,Brita/cimento máximo,OK
8,Cimento mínimo,OK
9,Cimento máximo,OK


Resultado da planilha: VIÁVEL.


## 2. Otimização em Python

Esta etapa calcula uma mistura mais econômica dentro das restrições do modelo. A lógica busca reduzir o cimento, que é o principal componente de custo, mantendo volume, fck, água/cimento e proporções viáveis.


In [4]:
# Cálculo direto de uma solução próxima do ótimo, considerando:
# volume mínimo = 0,995 m³
# relação água/cimento = 0,55
# areia/cimento = 2,53
# brita/cimento = 2,97
#
# volume = C/3150 + (0,55C)/1000 + (2,53C)/2650 + (2,97C)/2700 + volume_ar

denominador = (
    1 / massa_especifica["cimento"] +
    restricoes["ac_max"] / massa_especifica["agua"] +
    restricoes["areia_cimento_max"] / massa_especifica["areia"] +
    restricoes["brita_cimento_max"] / massa_especifica["brita"]
)

cimento_otimo = (restricoes["volume_min"] - volume_ar) / denominador
agua_otima = restricoes["ac_max"] * cimento_otimo
areia_otima = restricoes["areia_cimento_max"] * cimento_otimo
brita_otima = restricoes["brita_cimento_max"] * cimento_otimo

mistura_python = {
    "cimento": cimento_otimo,
    "agua": agua_otima,
    "areia": areia_otima,
    "brita": brita_otima
}

ind_python, checks_python = verificar_restricoes(**mistura_python)

df_python = pd.DataFrame([
    ["Cimento CP II-32", mistura_python["cimento"], "kg"],
    ["Água", mistura_python["agua"], "L"],
    ["Areia média", mistura_python["areia"], "kg"],
    ["Brita 1", mistura_python["brita"], "kg"],
], columns=["Material", "Quantidade", "Unidade"])

df_ind_python = pd.DataFrame(ind_python.items(), columns=["Indicador", "Valor"])
df_check_python = pd.DataFrame(
    [(k, "OK" if v else "FORA") for k, v in checks_python.items()],
    columns=["Restrição", "Atendida?"]
)

print("=== Mistura otimizada em Python ===")
display(df_python.round(4))
display(df_ind_python.round(6))
display(df_check_python)

if all(checks_python.values()):
    print("Resultado otimizado: VIÁVEL.")
else:
    print("Resultado otimizado: NÃO VIÁVEL.")


=== Mistura otimizada em Python ===


,Material,Quantidade,Unidade
0,Cimento CP II-32,333.6553,kg
1,Água,183.5104,L
2,Areia média,844.1480,kg
3,Brita 1,990.9563,kg


,Indicador,Valor
0,Custo total (R$/m³),319.47697
1,Volume calculado (m³),0.99500
2,Relação água/cimento,0.55000
3,fck estimado (MPa),25.00000
4,Areia/cimento,2.53000
5,Brita/cimento,2.97000


,Restrição,Atendida?
0,Volume mínimo,OK
1,Volume máximo,OK
2,Água/cimento máximo,OK
3,fck mínimo,OK
4,Areia/cimento mínimo,OK
5,Areia/cimento máximo,OK
6,Brita/cimento mínimo,OK
7,Brita/cimento máximo,OK
8,Cimento mínimo,OK
9,Cimento máximo,OK


Resultado otimizado: VIÁVEL.


## 3. Comparação entre planilha e Python


In [5]:
comparacao = pd.DataFrame([
    ["Cimento (kg)", mistura_planilha["cimento"], mistura_python["cimento"]],
    ["Água (L)", mistura_planilha["agua"], mistura_python["agua"]],
    ["Areia (kg)", mistura_planilha["areia"], mistura_python["areia"]],
    ["Brita (kg)", mistura_planilha["brita"], mistura_python["brita"]],
    ["Custo total (R$/m³)", ind_planilha["Custo total (R$/m³)"], ind_python["Custo total (R$/m³)"]],
    ["Volume calculado (m³)", ind_planilha["Volume calculado (m³)"], ind_python["Volume calculado (m³)"]],
    ["Relação água/cimento", ind_planilha["Relação água/cimento"], ind_python["Relação água/cimento"]],
    ["fck estimado (MPa)", ind_planilha["fck estimado (MPa)"], ind_python["fck estimado (MPa)"]],
], columns=["Item", "Planilha", "Python otimizado"])

comparacao["Diferença"] = comparacao["Python otimizado"] - comparacao["Planilha"]

display(comparacao.round(4))


,Item,Planilha,Python otimizado,Diferença
0,Cimento (kg),350.0000,333.6553,-16.3447
1,Água (L),190.0000,183.5104,-6.4896
2,Areia (kg),780.0000,844.1480,64.1480
3,Brita (kg),1030.0000,990.9563,-39.0437
4,Custo total (R$/m³),328.9910,319.4770,-9.5140
5,Volume calculado (m³),0.9969,0.9950,-0.0019
6,Relação água/cimento,0.5429,0.5500,0.0071
7,fck estimado (MPa),25.5618,25.0000,-0.5618


## 4. Conclusão

O modelo desenvolvido em Python teve como objetivo complementar a planilha feita no Excel com Solver, reproduzindo os principais cálculos utilizados na otimização da mistura de concreto. Foram considerados o custo total dos materiais, o volume calculado para 1 m³ de concreto, a relação água/cimento, o fck estimado e as proporções entre cimento, areia e brita.

A solução final obtida apresentou valores muito próximos nas duas ferramentas:

- Na planilha com Solver: **R$ 319,43/m³**

- Na simulação em Python: **R$ 319,47/m³**

A pequena diferença entre os resultados ocorre por causa de arredondamentos numéricos, casas decimais utilizadas nos cálculos e diferenças na forma como cada ferramenta processa os valores.

Mesmo com essa pequena variação, os dois resultados são considerados equivalentes para fins acadêmicos, pois indicam a mesma ordem de grandeza e confirmam a viabilidade da mistura otimizada. Em ambos os casos, a composição atende aos critérios definidos no modelo: volume próximo de 1 m³, fck mínimo de 25 MPa, relação água/cimento dentro do limite estabelecido e proporções viáveis entre os materiais.

A mistura otimizada ficou aproximadamente composta por:

- **Cimento:** 333,6 kg
- **Água:** 183,3 L
- **Areia:** 844,3 kg
- **Brita:** 990,1 kg

A análise também mostrou que o cimento continua sendo o material de maior impacto no custo total da mistura. Isso ocorre porque, além de ter maior preço unitário em relação aos demais materiais, ele influencia diretamente a resistência estimada do concreto.

Dessa forma, o uso do Solver no Excel e da simulação em Python demonstrou que ferramentas computacionais podem auxiliar na escolha de uma mistura mais econômica, mantendo requisitos mínimos de desempenho. No entanto, é importante destacar que o modelo possui caráter acadêmico e simplificado. Em uma aplicação real, a dosagem do concreto deveria ser validada por ensaios laboratoriais, considerando as características reais dos materiais, condições de cura, trabalhabilidade e exigências normativas específicas.
